# 00 — Data ingestion and validation
Tải raw data từ S3/local, tạo manifest và kiểm tra data contract. Logic nằm trong `src/cooling_load/ingestion` và `validation.py`.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path[:0] = [str(ROOT / 'src'), str(ROOT)]
from cooling_load.config import load_config
from cooling_load.pipeline import run_ingestion_and_validation
from cooling_load.synthetic import generate_synthetic_data


In [ ]:
config = load_config(ROOT / 'configs/base.yaml')
raw_path = config.path('raw_data')
if config.raw['ingestion']['mode'] == 'local' and not raw_path.exists():
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    generate_synthetic_data().to_csv(raw_path, index=False)
raw, validation_report = run_ingestion_and_validation(config)
validation_report


In [ ]:
display(raw.head())
display(raw.groupby('building_id').agg(rows=('timestamp','size'), start=('timestamp','min'), end=('timestamp','max')))
print('Manifest:', config.path('manifest'))
